# Mô Phỏng Máy Hút Bụi (Vacuum Cleaner Agent)

**Quy ước ma trận:**
- `0` → Ô trống (sạch) / Máy hút bụi
- `1` → Bụi

**Mở rộng:** Bổ sung thuật toán Uniform Cost Search (UCS) và Giao diện UI Animation minh họa cho từng bước đi.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import heapq
import itertools
import time
from IPython.display import clear_output


In [ ]:
# ── Cấu hình ──
ROWS      = 5
COLS      = 7
DUST_PROB = 0.4

# ── Tạo môi trường ──
def create_env(rows, cols, dust_prob):
    grid = np.zeros((rows, cols), dtype=int)
    for r in range(rows):
        for c in range(cols):
            if random.random() < dust_prob:
                grid[r][c] = 1
    return grid

# ── Vẽ ma trận ──
def draw_grid(grid, pos, title):
    rows, cols = grid.shape
    fig, ax = plt.subplots(figsize=(cols * 0.9, rows * 0.9))

    cmap = ListedColormap(['#F0F0F0', '#F4D03F'])  # 0=xám nhạt, 1=vàng
    ax.imshow(grid, cmap=cmap, vmin=0, vmax=1)

    for r in range(rows):
        for c in range(cols):
            if (r, c) == pos:
                ax.text(c, r, '🤖', ha='center', va='center', fontsize=14)
            elif grid[r][c] == 1:
                ax.text(c, r, '●', ha='center', va='center',
                        fontsize=16, color='#884400')
            else:
                ax.text(c, r, '0', ha='center', va='center',
                        fontsize=11, color='#888888')

    ax.set_xticks(np.arange(cols))
    ax.set_yticks(np.arange(rows))
    ax.set_xticklabels(np.arange(cols))
    ax.set_yticklabels(np.arange(rows))
    ax.set_xticks(np.arange(-0.5, cols, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, rows, 1), minor=True)
    ax.grid(which='minor', color='#AAAAAA', linewidth=0.8)
    ax.tick_params(which='minor', bottom=False, left=False)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=8)

    legend = [
        mpatches.Patch(color='#F0F0F0', label='0 - Ô sạch / Máy'),
        mpatches.Patch(color='#F4D03F', label='1 - Bụi'),
    ]
    ax.legend(handles=legend, loc='upper right',
              bbox_to_anchor=(1.35, 1.02), fontsize=9)

    plt.tight_layout()
    plt.show()


# ── Khởi tạo ──
grid = create_env(ROWS, COLS, DUST_PROB)
total_dust = int(np.sum(grid == 1))

print(f"Ma trận ban đầu  |  Tổng bụi: {total_dust} ô")
draw_grid(grid, pos=(-1, -1), title=f'Ma trận ban đầu — Bụi: {total_dust} ô')


In [ ]:
def get_path_to_dust_ucs(grid, start_pos):
    """
    Thuật toán Uniform Cost Search (UCS) để tìm đường đến ô có bụi.
    Sử dụng hàng đợi ưu tiên (Priority Queue / Min-Heap) để mở rộng các nút có chi phí thấp nhất.
    Trong trường hợp grid cơ bản, giả định chi phí cho mỗi bước di chuyển = 1.
    """
    rows, cols = grid.shape
    
    # Khởi tạo counter để giải quyết các phần tử có cost bằng nhau trong heap
    counter = itertools.count()
    
    # Priority queue chứa tuple: (cost, counter, current_pos, path)
    pq = []
    heapq.heappush(pq, (0, next(counter), start_pos, [start_pos]))
    
    # Dictionary lưu trữ chi phí nhỏ nhất (best cost) đến mỗi ô
    visited_costs = {start_pos: 0}
    
    while pq:
        cost, _, current_pos, path = heapq.heappop(pq)
        r, c = current_pos
        
        # Nếu chi phí lớn hơn chi phí nhỏ nhất đã biết, bỏ qua nút này (Late Goal Test & Cost Check)
        if cost > visited_costs.get(current_pos, float('inf')):
            continue
            
        # Goal test
        if grid[r][c] == 1:
            return path
            
        # Xét 4 hướng di chuyển: Lên, Xuống, Trái, Phải
        for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols:
                new_cost = cost + 1 # Chi phí di chuyển là 1 cho mỗi bước
                
                # Chỉ thêm vào frontier nếu tìm được đường đi tốt hơn (rẻ hơn)
                if new_cost < visited_costs.get((nr, nc), float('inf')):
                    visited_costs[(nr, nc)] = new_cost
                    heapq.heappush(pq, (new_cost, next(counter), (nr, nc), path + [(nr, nc)]))
                    
    return []


In [ ]:
def run_agent_ucs(grid_in):
    grid = grid_in.copy()
    rows, cols = grid.shape
    steps = 0
    cleaned = 0
    curr_pos = (0, 0)
    
    # Hiển thị UI ban đầu
    clear_output(wait=True)
    draw_grid(grid, pos=curr_pos, title=f'Bắt đầu tại {curr_pos}')
    time.sleep(0.5)
    
    if grid[curr_pos[0]][curr_pos[1]] == 1:
        grid[curr_pos[0]][curr_pos[1]] = 0
        cleaned += 1
        clear_output(wait=True)
        draw_grid(grid, pos=curr_pos, title=f'Phát hiện bụi! Hút bụi tại {curr_pos} — Đã hút: {cleaned}/{total_dust}')
        time.sleep(0.5)
        
    while cleaned < total_dust:
        path = get_path_to_dust_ucs(grid, curr_pos)
            
        if not path:
            print("Không thể tìm thấy thêm bụi nào!")
            break
            
        for next_pos in path[1:]:
            steps += 1
            r, c = next_pos
            pr, pc = curr_pos
            if r > pr: direction = 'XUỐNG'
            elif r < pr: direction = 'LÊN'
            elif c > pc: direction = 'PHẢI'
            else: direction = 'TRÁI'
            
            curr_pos = next_pos
            
            if grid[r][c] == 1:
                grid[r][c] = 0
                cleaned += 1
                action_msg = f'Bước {steps}: {direction} → Hút bụi tại ({r}, {c})\nĐã hút: {cleaned}/{total_dust}'
            else:
                action_msg = f'Bước {steps}: {direction} → ({r}, {c}) Ô sạch'
                
            # Vẽ hình UI di chuyển theo thời gian thực
            clear_output(wait=True)
            draw_grid(grid, pos=(r, c), title=action_msg)
            time.sleep(0.5)

    print('=' * 45)
    if cleaned == total_dust:
        status = 'THÀNH CÔNG'
        reason = 'Đã tìm và hút sạch hết bụi'
    else:
        status = 'THẤT BẠI'
        reason = 'Không hút hết bụi'

    print(f'Số bước đi  : {steps}')
    print(f'Bụi đã hút  : {cleaned} / {total_dust} ô')
    print(f'Trạng thái  : {status}')
    print(f'Lý do       : {reason}\n')


In [ ]:
run_agent_ucs(grid)
